<a href="https://colab.research.google.com/github/farazaghajani-eng/repowering_flexibility_optimization/blob/main/Two_Stage_Stochastic_Unit_Commitment_with_Storage_and_Load_Shedding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pulp import LpProblem, LpMinimize, LpVariable, lpSum, LpStatus, GLPK
from pulp import CPLEX_CMD
# Updated Generator Features
generators = [
    {"name": "gen05", "RampUp": 60, "RampDown": 60, "Startup": 100, "StartupCost": 110, "Pmax": 250, "Pmin": 100},
    {"name": "gen10", "RampUp": 60, "RampDown": 60, "Startup": 100, "StartupCost": 100, "Pmax": 250, "Pmin": 100},
    {"name": "gen11", "RampUp": 70, "RampDown": 70, "Startup": 100, "StartupCost": 100, "Pmax": 300, "Pmin": 100},
    {"name": "gen14", "RampUp": 60, "RampDown": 60, "Startup": 100, "StartupCost": 100, "Pmax": 250, "Pmin": 100},
    {"name": "gen19", "RampUp": 60, "RampDown": 60, "Startup": 100, "StartupCost": 100, "Pmax": 250, "Pmin": 100},
    {"name": "gen21", "RampUp": 50, "RampDown": 50, "Startup": 50, "StartupCost": 100, "Pmax": 250, "Pmin": 50},
    {"name": "gen23", "RampUp": 60, "RampDown": 60, "Startup": 100, "StartupCost": 100, "Pmax": 250, "Pmin": 100},
    {"name": "gen26", "RampUp": 60, "RampDown": 60, "Startup": 100, "StartupCost": 100, "Pmax": 250, "Pmin": 100},
    {"name": "gen27", "RampUp": 60, "RampDown": 60, "Startup": 100, "StartupCost": 100, "Pmax": 250, "Pmin": 100},
    {"name": "gen28", "RampUp": 60, "RampDown": 60, "Startup": 100, "StartupCost": 100, "Pmax": 250, "Pmin": 100},
    {"name": "gen29", "RampUp": 60, "RampDown": 60, "Startup": 80, "StartupCost": 100, "Pmax": 250, "Pmin": 80},
    {"name": "gen37", "RampUp": 60, "RampDown": 60, "Startup": 100, "StartupCost": 100, "Pmax": 250, "Pmin": 100},
    {"name": "gen39", "RampUp": 60, "RampDown": 60, "Startup": 100, "StartupCost": 100, "Pmax": 250, "Pmin": 100},
    {"name": "gen43", "RampUp": 60, "RampDown": 60, "Startup": 100, "StartupCost": 100, "Pmax": 250, "Pmin": 100},
    {"name": "gen44", "RampUp": 60, "RampDown": 60, "Startup": 100, "StartupCost": 100, "Pmax": 250, "Pmin": 100},
    {"name": "gen45", "RampUp": 60, "RampDown": 60, "Startup": 100, "StartupCost": 110, "Pmax": 250, "Pmin": 100},
    {"name": "gen52", "RampUp": 60, "RampDown": 60, "Startup": 100, "StartupCost": 110, "Pmax": 250, "Pmin": 100},
    {"name": "gen53", "RampUp": 60, "RampDown": 60, "Startup": 100, "StartupCost": 100, "Pmax": 250, "Pmin": 100},
]

storages = [
    {"name": "PSH1", "RampUp": 2, "RampDown": 2, "Startup": 200, "StartupCost": 10, "Pmax": 210, "Pmin": 100},
    {"name": "PSH2", "RampUp": 2, "RampDown": 2, "Startup": 200, "StartupCost": 10, "Pmax": 210, "Pmin": 100},
    {"name": "PSH3", "RampUp": 2, "RampDown": 2, "Startup": 200, "StartupCost": 10, "Pmax": 210, "Pmin": 100},
    {"name": "PSH4", "RampUp": 2, "RampDown": 2, "Startup": 200, "StartupCost": 10, "Pmax": 210, "Pmin": 100},
    {"name": "PSH5", "RampUp": 2, "RampDown": 2, "Startup": 200, "StartupCost": 10, "Pmax": 210, "Pmin": 100},
    {"name": "PSH6", "RampUp": 2, "RampDown": 2, "Startup": 200, "StartupCost": 10, "Pmax": 210, "Pmin": 100},
    {"name": "PSH7", "RampUp": 2, "RampDown": 2, "Startup": 200, "StartupCost": 10, "Pmax": 210, "Pmin": 100},
    {"name": "PSH8", "RampUp": 2, "RampDown": 2, "Startup": 200, "StartupCost": 10, "Pmax": 210, "Pmin": 100},
    {"name": "PSH9", "RampUp": 2, "RampDown": 2, "Startup": 100, "StartupCost": 10, "Pmax": 210, "Pmin": 100},
]
# Demand Upper Bound
demand = [
    3291.96, 2358.38, 1726.88, 3255.33,
    4119.83, 4427.13, 3555.79, 3028.29,
    4099.33, 5044.42, 4487.04, 3549.00
]
#wind of 118-bus IEEE
wind = 3

#Calculate the Net_load
net_load = [demand[i] - wind for i in range(len(demand))]
print(net_load)
# Create Optimization Problem
problem = LpProblem("GeneratorOptimization", LpMinimize)

# Decision Variables
power = {

    **{(g["name"], t): LpVariable(f"power_{g['name']}_{t}", 0, g["Pmax"])
       for g in generators for t in range(len(demand))},
    **{(s["name"], t): LpVariable(f"power_{s['name']}_{t}", 0, s["Pmax"])
       for s in storages for t in range(len(demand))}
}


status = {
    **{(g["name"], t): LpVariable(f"status_{g['name']}_{t}", 0, 1, cat="Binary")
    for g in generators for t in range(len(demand))},
     **{(s["name"], t): LpVariable(f"status_{s['name']}_{t}", 0, 1, cat="Binary")
    for s in storages for t in range(len(demand))}
}

#the estimation of Investment cost of gen26,27,28
Investment_cost = 60404925

# Objective Function: Minimize Cost
problem += lpSum(
    power[g["name"], t] * 0.05 + status[g["name"], t] * g["StartupCost"] + Investment_cost


    # The next line ensures t+1 does not cause an Index Error by excluding the last element of the range
    for g in generators for t in range(len(demand))
) + lpSum(
    power[s["name"], t] * 0.1 + status[s["name"], t] * s["StartupCost"]
    for s in storages for t in range(len(demand))
)

# Constraints
for t in range(len(demand)):
    # Meet Demand
    problem += lpSum(power[g["name"], t] for g in generators) + lpSum(power[s["name"], t] for s in storages) >= net_load[t], f"Demand_{t}"

    # Separate loops for generator and storage constraints:
    for g in generators:
        name_1 = g["name"]
        # Enforce generator minimum and maximum if on
        problem += power[name_1, t] <= status[name_1, t] * g["Pmax"], f"MaxPower_{name_1}_{t}"
        problem += power[name_1, t] >= status[name_1, t] * g["Pmin"], f"MinPower_{name_1}_{t}"

        # Ramp-up and ramp-down constraints for generators
        if t > 0:
            problem += power[name_1, t] - power[name_1, t - 1] <= g["RampUp"], f"RampUp_{name_1}_{t}"
            problem += power[name_1, t - 1] - power[name_1, t] <= g["RampDown"], f"RampDown_{name_1}_{t}"

    for s in storages:
        name_2 = s["name"]
        # Enforce storage minimum and maximum if on
        problem += power[name_2, t] <= status[name_2, t] * s["Pmax"], f"MaxPower_{name_2}_{t}"
        problem += power[name_2, t] >= status[name_2, t] * s["Pmin"], f"MinPower_{name_2}_{t}"

        # Ramp-up and ramp-down constraints for storages
        if t > 0:
            problem += power[name_2, t] - power[name_2, t - 1] <= s["RampUp"], f"RampUp_{name_2}_{t}"
            problem += power[name_2, t - 1] - power[name_2, t] <= s["RampDown"], f"RampDown_{name_2}_{t}"

# Power constraint at t=0 (startup capacity constraints)
    if t == 0:
        for g in generators:
            problem += power[g["name"], t] <= g["Startup"], f"Startup_Constraint_t0_{g['name']}"

# Power constraint at t=0 (startup capacity constraints)
    if t == 0:
        for s in storages:
            problem += power[s["name"], t] <= s["Startup"], f"Startup_Constraint_t0_{s['name']}"

# Solve Problem
problem.solve(GLPK())
# Get the optimal cost
if LpStatus[problem.status] == "Optimal":
    optimal_cost = problem.objective.value()
    print(f"The optimal cost is: {optimal_cost}")
else:
    print(f"The problem could not be solved optimally. Status: {LpStatus[problem.status]}")
# Output Results
for g in generators:
    for t in range(len(demand)):
        print(f"{g['name']} at time {t}: Power = {power[g['name'], t].varValue}, Status = {status[g['name'], t].varValue}")
for s in storages:
    for t in range(len(demand)):
        print(f"{s['name']} at time {t}: Power = {power[s['name'], t].varValue}, Status = {status[s['name'], t].varValue}")

[3288.96, 2355.38, 1723.88, 3252.33, 4116.83, 4424.13, 3552.79, 3025.29, 4096.33, 5041.42, 4484.04, 3546.0]
The optimal cost is: 13047487634.929993
gen05 at time 0: Power = 100.0, Status = 1
gen05 at time 1: Power = 100.0, Status = 1
gen05 at time 2: Power = 100.0, Status = 1
gen05 at time 3: Power = 160.0, Status = 1
gen05 at time 4: Power = 220.0, Status = 1
gen05 at time 5: Power = 160.0, Status = 1
gen05 at time 6: Power = 100.0, Status = 1
gen05 at time 7: Power = 100.0, Status = 1
gen05 at time 8: Power = 160.0, Status = 1
gen05 at time 9: Power = 220.0, Status = 1
gen05 at time 10: Power = 160.0, Status = 1
gen05 at time 11: Power = 100.0, Status = 1
gen10 at time 0: Power = 100.0, Status = 1
gen10 at time 1: Power = 100.0, Status = 1
gen10 at time 2: Power = 100.0, Status = 1
gen10 at time 3: Power = 100.0, Status = 1
gen10 at time 4: Power = 160.0, Status = 1
gen10 at time 5: Power = 100.0, Status = 1
gen10 at time 6: Power = 139.83, Status = 1
gen10 at time 7: Power = 100.0, 

In [ ]:
# @title AI prompt cell

import ipywidgets as widgets
from IPython.display import display, HTML, Markdown,clear_output
from google.colab import ai

dropdown = widgets.Dropdown(
    options=[],
    layout={'width': 'auto'}
)

def update_model_list(new_options):
    dropdown.options = new_options
update_model_list(ai.list_models())

text_input = widgets.Textarea(
    placeholder='Ask me anything....',
    layout={'width': 'auto', 'height': '100px'},
)

button = widgets.Button(
    description='Submit Text',
    disabled=False,
    tooltip='Click to submit the text',
    icon='check'
)

output_area = widgets.Output(
     layout={'width': 'auto', 'max_height': '300px','overflow_y': 'scroll'}
)

def on_button_clicked(b):
    with output_area:
        output_area.clear_output(wait=False)
        accumulated_content = ""
        for new_chunk in ai.generate_text(prompt=text_input.value, model_name=dropdown.value, stream=True):
            if new_chunk is None:
                continue
            accumulated_content += new_chunk
            clear_output(wait=True)
            display(Markdown(accumulated_content))

button.on_click(on_button_clicked)
vbox = widgets.GridBox([dropdown, text_input, button, output_area])

display(HTML("""
<style>
.widget-dropdown select {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
.widget-textarea textarea {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
</style>
"""))
display(vbox)


In [2]:
# Install
!pip install pyomo
!apt-get install -y -qq glpk-utils

import pyomo.environ as pyo

# -----------------------------
# SETS
# -----------------------------
T = range(12)
G = ["gen1", "gen2"]
S = ["PSH1"]

Omega = ["low", "medium", "high"]
prob = {"low": 0.3, "medium": 0.5, "high": 0.2}

# -----------------------------
# PARAMETERS
# -----------------------------
Pmax = {"gen1": 250, "gen2": 300}
Pmin = {"gen1": 100, "gen2": 100}
RampUp = {"gen1": 60, "gen2": 70}
RampDown = {"gen1": 60, "gen2": 70}
Cost = {"gen1": 20, "gen2": 25}
StartupCost = {"gen1": 100, "gen2": 120}

# Storage
Pmax_s = {"PSH1": 200}
Emax = {"PSH1": 500}
Eff_c = {"PSH1": 0.9}
Eff_d = {"PSH1": 0.9}

# Demand
demand = {
    (w, t): (2000 + 50 * t if w == "low" else
             2500 + 50 * t if w == "medium" else
             3000 + 50 * t)
    for w in Omega for t in T
}

VOLL = 10000  # penalty for unmet demand

# -----------------------------
# MODEL
# -----------------------------
m = pyo.ConcreteModel()

m.T = pyo.Set(initialize=T)
m.G = pyo.Set(initialize=G)
m.S = pyo.Set(initialize=S)
m.Omega = pyo.Set(initialize=Omega)

# -----------------------------
# VARIABLES
# -----------------------------
# Commitment
m.u = pyo.Var(m.G, m.T, within=pyo.Binary)
m.su = pyo.Var(m.G, m.T, within=pyo.Binary)

# Dispatch
m.p = pyo.Var(m.G, m.T, m.Omega, within=pyo.NonNegativeReals)

# Storage
m.charge = pyo.Var(m.S, m.T, m.Omega, within=pyo.NonNegativeReals)
m.discharge = pyo.Var(m.S, m.T, m.Omega, within=pyo.NonNegativeReals)
m.soc = pyo.Var(m.S, m.T, m.Omega, within=pyo.NonNegativeReals)

# Load shedding (KEY FIX)
m.lshed = pyo.Var(m.T, m.Omega, within=pyo.NonNegativeReals)

# -----------------------------
# OBJECTIVE
# -----------------------------
def obj_rule(m):
    return (
        # Startup
        sum(StartupCost[g] * m.su[g, t] for g in m.G for t in m.T)
        +
        # Expected cost
        sum(
            prob[w] * (
                sum(Cost[g] * m.p[g, t, w] for g in m.G for t in m.T)
                + VOLL * sum(m.lshed[t, w] for t in m.T)
            )
            for w in m.Omega
        )
    )

m.obj = pyo.Objective(rule=obj_rule, sense=pyo.minimize)

# -----------------------------
# CONSTRAINTS
# -----------------------------

# Generator limits
def gen_max(m, g, t, w):
    return m.p[g, t, w] <= Pmax[g] * m.u[g, t]
m.gen_max = pyo.Constraint(m.G, m.T, m.Omega, rule=gen_max)

def gen_min(m, g, t, w):
    return m.p[g, t, w] >= Pmin[g] * m.u[g, t]
m.gen_min = pyo.Constraint(m.G, m.T, m.Omega, rule=gen_min)

# Ramping
def ramp_up(m, g, t, w):
    if t == 0:
        return pyo.Constraint.Skip
    return m.p[g, t, w] - m.p[g, t-1, w] <= RampUp[g]
m.ramp_up = pyo.Constraint(m.G, m.T, m.Omega, rule=ramp_up)

def ramp_down(m, g, t, w):
    if t == 0:
        return pyo.Constraint.Skip
    return m.p[g, t-1, w] - m.p[g, t, w] <= RampDown[g]
m.ramp_down = pyo.Constraint(m.G, m.T, m.Omega, rule=ramp_down)

# Startup logic
def startup(m, g, t):
    if t == 0:
        return m.su[g, t] >= m.u[g, t]
    return m.su[g, t] >= m.u[g, t] - m.u[g, t-1]
m.startup = pyo.Constraint(m.G, m.T, rule=startup)

# Demand balance (FIXED)
def balance(m, t, w):
    return (
        sum(m.p[g, t, w] for g in m.G)
        + sum(m.discharge[s, t, w] for s in m.S)
        - sum(m.charge[s, t, w] for s in m.S)
        + m.lshed[t, w]   # ensures feasibility
        >= demand[(w, t)]
    )
m.balance = pyo.Constraint(m.T, m.Omega, rule=balance)

# Storage power limits
def charge_limit(m, s, t, w):
    return m.charge[s, t, w] <= Pmax_s[s]
m.charge_lim = pyo.Constraint(m.S, m.T, m.Omega, rule=charge_limit)

def discharge_limit(m, s, t, w):
    return m.discharge[s, t, w] <= Pmax_s[s]
m.discharge_lim = pyo.Constraint(m.S, m.T, m.Omega, rule=discharge_limit)

# SOC dynamics
def soc_rule(m, s, t, w):
    if t == 0:
        return m.soc[s, t, w] == 100
    return m.soc[s, t, w] == (
        m.soc[s, t-1, w]
        + Eff_c[s] * m.charge[s, t, w]
        - m.discharge[s, t, w] / Eff_d[s]
    )
m.soc_dyn = pyo.Constraint(m.S, m.T, m.Omega, rule=soc_rule)

# SOC bounds
def soc_lim(m, s, t, w):
    return m.soc[s, t, w] <= Emax[s]
m.soc_lim = pyo.Constraint(m.S, m.T, m.Omega, rule=soc_lim)

# -----------------------------
# SOLVE
# -----------------------------
solver = pyo.SolverFactory("glpk")
results = solver.solve(m)

# -----------------------------
# OUTPUT
# -----------------------------
print("Status:", results.solver.termination_condition)
print("Objective:", pyo.value(m.obj))

print("\nLoad shedding (critical insight):")
for t in m.T:
    for w in m.Omega:
        val = pyo.value(m.lshed[t, w])
        if val > 1e-3:
            print(f"t={t}, scenario={w}, shed={val:.2f}")

Selecting previously unselected package libsuitesparseconfig5:amd64.
(Reading database ... 118194 files and directories currently installed.)
Preparing to unpack .../libsuitesparseconfig5_1%3a5.10.1+dfsg-4build1_amd64.deb ...
Unpacking libsuitesparseconfig5:amd64 (1:5.10.1+dfsg-4build1) ...
Selecting previously unselected package libamd2:amd64.
Preparing to unpack .../libamd2_1%3a5.10.1+dfsg-4build1_amd64.deb ...
Unpacking libamd2:amd64 (1:5.10.1+dfsg-4build1) ...
Selecting previously unselected package libcolamd2:amd64.
Preparing to unpack .../libcolamd2_1%3a5.10.1+dfsg-4build1_amd64.deb ...
Unpacking libcolamd2:amd64 (1:5.10.1+dfsg-4build1) ...
Selecting previously unselected package libglpk40:amd64.
Preparing to unpack .../libglpk40_5.0-1_amd64.deb ...
Unpacking libglpk40:amd64 (5.0-1) ...
Selecting previously unselected package glpk-utils.
Preparing to unpack .../glpk-utils_5.0-1_amd64.deb ...
Unpacking glpk-utils (5.0-1) ...
Setting up libsuitesparseconfig5:amd64 (1:5.10.1+dfsg-4b

In [1]:
lshed[t,w]


NameError: name 'lshed' is not defined

In [ ]:
# Install Pyomo + solver
!pip install pyomo
!apt-get install -y -qq glpk-utils

import pyomo.environ as pyo

# -----------------------------
# SETS
# -----------------------------
T = range(12)
G = ["gen1", "gen2"]
S = ["PSH1"]

# Scenarios (uncertainty in wind/load)
Omega = ["low", "medium", "high"]
prob = {"low": 0.3, "medium": 0.5, "high": 0.2}

# -----------------------------
# PARAMETERS
# -----------------------------
Pmax = {"gen1": 250, "gen2": 300}
Pmin = {"gen1": 100, "gen2": 100}
RampUp = {"gen1": 60, "gen2": 70}
RampDown = {"gen1": 60, "gen2": 70}
Cost = {"gen1": 20, "gen2": 25}
StartupCost = {"gen1": 100, "gen2": 120}

# Storage
Pmax_s = {"PSH1": 200}
Emax = {"PSH1": 500}
Eff_c = {"PSH1": 0.9}
Eff_d = {"PSH1": 0.9}

# Demand scenarios
demand = {
    (w, t): (2000 + 50 * t if w == "low" else
             2500 + 50 * t if w == "medium" else
             3000 + 50 * t)
    for w in Omega for t in T
}

# -----------------------------
# MODEL
# -----------------------------
m = pyo.ConcreteModel()

m.T = pyo.Set(initialize=T)
m.G = pyo.Set(initialize=G)
m.S = pyo.Set(initialize=S)
m.Omega = pyo.Set(initialize=Omega)

# -----------------------------
# VARIABLES
# -----------------------------
# First-stage (commitment)
m.u = pyo.Var(m.G, m.T, within=pyo.Binary)
m.su = pyo.Var(m.G, m.T, within=pyo.Binary)

# Second-stage (scenario dependent)
m.p = pyo.Var(m.G, m.T, m.Omega, within=pyo.NonNegativeReals)

# Storage
m.charge = pyo.Var(m.S, m.T, m.Omega, within=pyo.NonNegativeReals)
m.discharge = pyo.Var(m.S, m.T, m.Omega, within=pyo.NonNegativeReals)
m.soc = pyo.Var(m.S, m.T, m.Omega, within=pyo.NonNegativeReals)

# -----------------------------
# OBJECTIVE
# -----------------------------
def obj_rule(m):
    return (
        # Startup cost (first stage)
        sum(StartupCost[g] * m.su[g, t] for g in m.G for t in m.T)
        +
        # Expected operating cost
        sum(
            prob[w] * (
                sum(Cost[g] * m.p[g, t, w] for g in m.G for t in m.T)
            )
            for w in m.Omega
        )
    )

m.obj = pyo.Objective(rule=obj_rule, sense=pyo.minimize)

# -----------------------------
# CONSTRAINTS
# -----------------------------

# Power limits
def gen_limits(m, g, t, w):
    return m.p[g, t, w] <= Pmax[g] * m.u[g, t]
m.gen_max = pyo.Constraint(m.G, m.T, m.Omega, rule=gen_limits)

def gen_min(m, g, t, w):
    return m.p[g, t, w] >= Pmin[g] * m.u[g, t]
m.gen_min = pyo.Constraint(m.G, m.T, m.Omega, rule=gen_min)

# Ramp constraints
def ramp_up(m, g, t, w):
    if t == 0:
        return pyo.Constraint.Skip
    return m.p[g, t, w] - m.p[g, t-1, w] <= RampUp[g]
m.ramp_up = pyo.Constraint(m.G, m.T, m.Omega, rule=ramp_up)

def ramp_down(m, g, t, w):
    if t == 0:
        return pyo.Constraint.Skip
    return m.p[g, t-1, w] - m.p[g, t, w] <= RampDown[g]
m.ramp_down = pyo.Constraint(m.G, m.T, m.Omega, rule=ramp_down)

# Startup logic
def startup_logic(m, g, t):
    if t == 0:
        return m.su[g, t] >= m.u[g, t]
    return m.su[g, t] >= m.u[g, t] - m.u[g, t-1]
m.startup = pyo.Constraint(m.G, m.T, rule=startup_logic)

# Demand balance
def balance(m, t, w):
    return (
        sum(m.p[g, t, w] for g in m.G)
        + sum(m.discharge[s, t, w] for s in m.S)
        - sum(m.charge[s, t, w] for s in m.S)
        >= demand[(w, t)]
    )
m.balance = pyo.Constraint(m.T, m.Omega, rule=balance)

# Storage SOC
def soc_rule(m, s, t, w):
    if t == 0:
        return m.soc[s, t, w] == 100  # initial SOC
    return m.soc[s, t, w] == (
        m.soc[s, t-1, w]
        + Eff_c[s] * m.charge[s, t, w]
        - m.discharge[s, t, w] / Eff_d[s]
    )
m.soc_dyn = pyo.Constraint(m.S, m.T, m.Omega, rule=soc_rule)

# SOC limits
def soc_limit(m, s, t, w):
    return m.soc[s, t, w] <= Emax[s]
m.soc_lim = pyo.Constraint(m.S, m.T, m.Omega, rule=soc_limit)

# -----------------------------
# SOLVE
# -----------------------------
solver = pyo.SolverFactory("glpk")
results = solver.solve(m)

# -----------------------------
# OUTPUT
# -----------------------------
print(results.solver.status)
print(results.solver.termination_condition)
print("Objective:", pyo.value(m.obj))

Selecting previously unselected package libsuitesparseconfig5:amd64.
(Reading database ... 118194 files and directories currently installed.)
Preparing to unpack .../libsuitesparseconfig5_1%3a5.10.1+dfsg-4build1_amd64.deb ...
Unpacking libsuitesparseconfig5:amd64 (1:5.10.1+dfsg-4build1) ...
Selecting previously unselected package libamd2:amd64.
Preparing to unpack .../libamd2_1%3a5.10.1+dfsg-4build1_amd64.deb ...
Unpacking libamd2:amd64 (1:5.10.1+dfsg-4build1) ...
Selecting previously unselected package libcolamd2:amd64.
Preparing to unpack .../libcolamd2_1%3a5.10.1+dfsg-4build1_amd64.deb ...
Unpacking libcolamd2:amd64 (1:5.10.1+dfsg-4build1) ...
Selecting previously unselected package libglpk40:amd64.
Preparing to unpack .../libglpk40_5.0-1_amd64.deb ...
Unpacking libglpk40:amd64 (5.0-1) ...
Selecting previously unselected package glpk-utils.
Preparing to unpack .../glpk-utils_5.0-1_amd64.deb ...
Unpacking glpk-utils (5.0-1) ...
Setting up libsuitesparseconfig5:amd64 (1:5.10.1+dfsg-4b

ERROR:pyomo.common.numeric_types:evaluating object as numeric value: p[gen1,0,low]
    (object: <class 'pyomo.core.base.var.VarData'>)
No value for uninitialized VarData object p[gen1,0,low]
ERROR:pyomo.common.numeric_types:evaluating object as numeric value: obj
    (object: <class 'pyomo.core.base.objective.ScalarObjective'>)
No value for uninitialized VarData object p[gen1,0,low]


ok
infeasible


ValueError: No value for uninitialized VarData object p[gen1,0,low]